# Lab: LSTM Machine Translation + BLEU, ROUGE, BERTScore and MoverScore

**Task:** Train a small English ↔ Hindi machine-translation model with an encoder–decoder LSTM, generate translations, and evaluate them with BLEU, ROUGE, BERTScore, and MoverScore.

> **Important:** This is a classroom demonstration using a tiny, hand-written parallel dataset. Scores on the same sentences used for training are not evidence of general translation quality. For a meaningful evaluation, use a larger parallel corpus and a held-out test set.


## 1. Install packages

Run this cell in Google Colab. BERTScore and MoverScore use pretrained language representations, so their first run may download model files. MoverScore is an older package and can be sensitive to dependency versions; if that cell fails, the BLEU/ROUGE/BERTScore parts can still be run independently.


In [1]:
!pip -q install rouge-score nltk bert-score
!pip -q install moverscore


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 5.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.6/129.6 kB 10.1 MB/s eta 0:00:00


## 2. Imports and reproducibility

In [2]:
import os, random, re, numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Attention, Concatenate
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

random.seed(42)
np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow:", tf.__version__)


TensorFlow: 2.21.0


## 3. Prepare a tiny parallel corpus

This example translates **Hindi → English**. Each Hindi sentence is paired with its English reference. You can reverse the source and target columns to experiment with English → Hindi.


In [3]:
pairs = [
    ("नमस्ते", "hello"),
    ("आप कैसे हैं", "how are you"),
    ("मैं ठीक हूँ", "i am fine"),
    ("मेरा नाम राहुल है", "my name is rahul"),
    ("मैं एक छात्र हूँ", "i am a student"),
    ("मुझे पानी चाहिए", "i need water"),
    ("धन्यवाद", "thank you"),
    ("शुभ प्रभात", "good morning"),
    ("शुभ रात्रि", "good night"),
    ("मुझे खाना पसंद है", "i like food"),
    ("मैं घर जा रहा हूँ", "i am going home"),
    ("वह मेरी दोस्त है", "she is my friend"),
    ("यह एक किताब है", "this is a book"),
    ("दरवाज़ा खोलो", "open the door"),
    ("कृपया बैठिए", "please sit down"),
    ("मुझे मदद चाहिए", "i need help"),
    ("आज मौसम अच्छा है", "the weather is nice today"),
    ("मैं अंग्रेज़ी सीख रहा हूँ", "i am learning english"),
    ("आपका स्वागत है", "you are welcome"),
    ("फिर मिलेंगे", "see you again"),
    ("मुझे कॉफी पसंद है", "i like coffee"),
    ("वह स्कूल जाता है", "he goes to school"),
    ("हम दोस्त हैं", "we are friends"),
    ("बिल्ली सो रही है", "the cat is sleeping"),
    ("मैं बहुत खुश हूँ", "i am very happy"),
    ("धीरे बोलिए", "speak slowly"),
    ("यह कितना है", "how much is this"),
    ("मुझे समझ नहीं आया", "i did not understand"),
    ("बस स्टेशन कहाँ है", "where is the bus station"),
    ("मैं कल आऊँगा", "i will come tomorrow"),
    ("पानी गर्म है", "the water is hot"),
    ("अपना ध्यान रखना", "take care of yourself"),
]


source_texts = [s.strip() for s, t in pairs]
target_texts = ["<start> " + t.strip() + " <end>" for s, t in pairs]

print("Parallel pairs:", len(pairs))
for src, tgt in pairs[:5]:
    print(f"{src}  ->  {tgt}")


Parallel pairs: 32
नमस्ते  ->  hello
आप कैसे हैं  ->  how are you
मैं ठीक हूँ  ->  i am fine
मेरा नाम राहुल है  ->  my name is rahul
मैं एक छात्र हूँ  ->  i am a student


## 4. Tokenize and pad sequences

Separate tokenizers are used for source and target language. Padding is added so that each batch has the same sequence length.


In [4]:
src_tokenizer = Tokenizer(filters="", lower=False, oov_token="<unk>")
tgt_tokenizer = Tokenizer(filters="", lower=True, oov_token="<unk>")

src_tokenizer.fit_on_texts(source_texts)
tgt_tokenizer.fit_on_texts(target_texts)

src_sequences = src_tokenizer.texts_to_sequences(source_texts)
tgt_sequences = tgt_tokenizer.texts_to_sequences(target_texts)

max_src_len = max(len(x) for x in src_sequences)
max_tgt_len = max(len(x) for x in tgt_sequences)

encoder_input_data = pad_sequences(src_sequences, maxlen=max_src_len, padding="post")
target_full = pad_sequences(tgt_sequences, maxlen=max_tgt_len, padding="post")

decoder_input_data = target_full[:, :-1]
decoder_target_data = target_full[:, 1:]

src_vocab_size = len(src_tokenizer.word_index) + 1
tgt_vocab_size = len(tgt_tokenizer.word_index) + 1

print("Source vocabulary:", src_vocab_size)
print("Target vocabulary:", tgt_vocab_size)
print("Encoder input shape:", encoder_input_data.shape)
print("Decoder input shape:", decoder_input_data.shape)


Source vocabulary: 70
Target vocabulary: 74
Encoder input shape: (32, 5)
Decoder input shape: (32, 6)


## 5. Build an encoder–decoder LSTM with attention

The encoder produces a sequence of hidden states. The decoder LSTM produces target-side states. An attention layer lets each decoder position focus on the encoder sequence before predicting the next target token.


In [5]:
EMBED_DIM = 64
LATENT_DIM = 128

# Encoder
encoder_inputs = Input(shape=(None,), name="encoder_inputs")
enc_embedding_layer = Embedding(src_vocab_size, EMBED_DIM, mask_zero=True, name="encoder_embedding")
enc_emb = enc_embedding_layer(encoder_inputs)
encoder_lstm = LSTM(LATENT_DIM, return_sequences=True, return_state=True, name="encoder_lstm")
encoder_outputs, state_h, state_c = encoder_lstm(enc_emb)

# Decoder
decoder_inputs = Input(shape=(None,), name="decoder_inputs")
dec_embedding_layer = Embedding(tgt_vocab_size, EMBED_DIM, mask_zero=True, name="decoder_embedding")
dec_emb = dec_embedding_layer(decoder_inputs)
decoder_lstm = LSTM(LATENT_DIM, return_sequences=True, return_state=True, name="decoder_lstm")
decoder_outputs, _, _ = decoder_lstm(dec_emb, initial_state=[state_h, state_c])

attention_layer = Attention(name="decoder_attention")
context = attention_layer([decoder_outputs, encoder_outputs])
combined = Concatenate(name="context_and_decoder")([decoder_outputs, context])
output_layer = Dense(tgt_vocab_size, activation="softmax", name="token_probabilities")
decoder_probs = output_layer(combined)

model = Model([encoder_inputs, decoder_inputs], decoder_probs)
model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["sparse_categorical_accuracy"])
model.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_inputs      │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_embedding   │ (None, None, 64)  │      4,480 │ encoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, None)      │          0 │ encoder_inputs[0… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_embedding   │ (None, None, 64)  │      4,736 │ decoder_inputs[0… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ encoder_lstm (LSTM) │ [(None, None,     │     98,816 │ encoder_embeddin… │
│                     │ 128), (None,      │            │ not_equal[0][0]   │
│                     │ 128), (None,      │            │                   │
│                     │ 128)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_lstm (LSTM) │ [(None, None,     │     98,816 │ decoder_embeddin… │
│                     │ 128), (None,      │            │ encoder_lstm[0][… │
│                     │ 128), (None,      │            │ encoder_lstm[0][… │
│                     │ 128)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_attention   │ (None, None, 128) │          0 │ decoder_lstm[0][… │
│ (Attention)         │                   │            │ encoder_lstm[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ context_and_decoder │ (None, None, 256) │          0 │ decoder_lstm[0][… │
│ (Concatenate)       │                   │            │ decoder_attentio… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ token_probabilities │ (None, None, 74)  │     19,018 │ context_and_deco… │
│ (Dense)             │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 225,866 (882.29 KB)

 Trainable params: 225,866 (882.29 KB)

 Non-trainable params: 0 (0.00 B)

## 6. Train the model

The loss is masked for padding token ID `0`, so padded positions do not contribute to training loss. This tiny dataset is intentionally for demonstrating the pipeline, not for producing a robust translator.


In [6]:
# Create sample weights: ignore padding positions in the target.
sample_weights = (decoder_target_data != 0).astype("float32")
history = model.fit(
    [encoder_input_data, decoder_input_data],
    decoder_target_data,
    sample_weight=sample_weights,
    batch_size=8,
    epochs=250,
    verbose=0
)
print("Final training loss:", round(float(history.history["loss"][-1]), 4))


Final training loss: 0.0045


## 7. Build inference models and translate

At inference time, generate one token at a time. The inference decoder uses the same trained embedding, LSTM, attention and output layers.


In [7]:
# Encoder inference model returns the full encoder sequence plus final states.
encoder_model = Model(encoder_inputs, [encoder_outputs, state_h, state_c])

# One-step decoder inputs: current token, previous states, and encoder outputs.
token_input = Input(shape=(1,), name="token_input")
enc_seq_input = Input(shape=(max_src_len, LATENT_DIM), name="encoder_sequence_input")
state_h_input = Input(shape=(LATENT_DIM,), name="state_h_input")
state_c_input = Input(shape=(LATENT_DIM,), name="state_c_input")

token_emb = dec_embedding_layer(token_input)
step_output, next_h, next_c = decoder_lstm(token_emb, initial_state=[state_h_input, state_c_input])
step_context = attention_layer([step_output, enc_seq_input])
step_combined = Concatenate()([step_output, step_context])
step_probs = output_layer(step_combined)

decoder_model = Model(
    [token_input, enc_seq_input, state_h_input, state_c_input],
    [step_probs, next_h, next_c]
)

start_id = tgt_tokenizer.word_index["<start>"]
end_id = tgt_tokenizer.word_index["<end>"]
id_to_target = {idx: word for word, idx in tgt_tokenizer.word_index.items()}

def translate(sentence, max_steps=max_tgt_len):
    seq = src_tokenizer.texts_to_sequences([sentence.strip()])
    enc_input = pad_sequences(seq, maxlen=max_src_len, padding="post")
    enc_seq, h, c = encoder_model.predict(enc_input, verbose=0)

    current_id = start_id
    output_words = []
    for _ in range(max_steps):
        probs, h, c = decoder_model.predict(
            [np.array([[current_id]]), enc_seq, h, c], verbose=0
        )
        next_id = int(np.argmax(probs[0, -1, :]))
        if next_id == end_id or next_id == 0:
            break
        word = id_to_target.get(next_id, "")
        if word and word not in ("<start>", "<end>", "<unk>"):
            output_words.append(word)
        current_id = next_id
    return " ".join(output_words).strip()

for src, ref in pairs[:10]:
    print("Hindi:", src)
    print("Reference:", ref)
    print("Prediction:", translate(src))
    print("-" * 50)


Hindi: नमस्ते
Reference: hello
Prediction: hello
--------------------------------------------------
Hindi: आप कैसे हैं
Reference: how are you
Prediction: how are you
--------------------------------------------------
Hindi: मैं ठीक हूँ
Reference: i am fine
Prediction: i am fine
--------------------------------------------------
Hindi: मेरा नाम राहुल है
Reference: my name is rahul
Prediction: my name is rahul
--------------------------------------------------
Hindi: मैं एक छात्र हूँ
Reference: i am a student
Prediction: i am a student
--------------------------------------------------
Hindi: मुझे पानी चाहिए
Reference: i need water
Prediction: i need water
--------------------------------------------------
Hindi: धन्यवाद
Reference: thank you
Prediction: thank you
--------------------------------------------------
Hindi: शुभ प्रभात
Reference: good morning
Prediction: good morning
--------------------------------------------------
Hindi: शुभ रात्रि
Reference: good night
Prediction: good ni

## 8. Compute BLEU and ROUGE

BLEU measures n-gram precision with a brevity penalty. ROUGE measures overlap (here ROUGE-1, ROUGE-2 and ROUGE-L). We evaluate on the same small dataset used for training only to demonstrate the metric code; report this clearly in your lab submission.


In [8]:
import nltk
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from rouge_score import rouge_scorer

references = [t for s, t in pairs]
predictions = [translate(s) for s, t in pairs]

# BLEU expects tokenized references and hypotheses.
tokenized_references = [[ref.lower().split()] for ref in references]
tokenized_predictions = [pred.lower().split() for pred in predictions]
bleu = corpus_bleu(
    tokenized_references,
    tokenized_predictions,
    smoothing_function=SmoothingFunction().method1
)
print(f"Corpus BLEU: {bleu:.4f}")

rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
rouge_rows = []
for ref, pred in zip(references, predictions):
    scores = rouge.score(ref, pred)
    rouge_rows.append({k: scores[k].fmeasure for k in ["rouge1", "rouge2", "rougeL"]})

for metric in ["rouge1", "rouge2", "rougeL"]:
    avg = float(np.mean([row[metric] for row in rouge_rows]))
    print(f"{metric.upper()} average F1: {avg:.4f}")


Corpus BLEU: 0.8281
ROUGE1 average F1: 1.0000
ROUGE2 average F1: 0.9688
ROUGEL average F1: 1.0000


## 9. Compute BERTScore

BERTScore uses contextual embeddings from a pretrained language model instead of exact n-gram matches. Since the generated target language in this notebook is English, use an English model (`roberta-large`). If you swap the direction to Hindi output, select a suitable multilingual model such as `bert-base-multilingual-cased`.


In [9]:
from bert_score import score as bert_score

# English target: use an English pretrained model.
P, R, F1 = bert_score(
    predictions,
    references,
    lang="en",
    model_type="roberta-large",
    verbose=True
)
print(f"BERTScore Precision: {P.mean().item():.4f}")
print(f"BERTScore Recall:    {R.mean().item():.4f}")
print(f"BERTScore F1:        {F1.mean().item():.4f}")


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 1.63 seconds, 19.59 sentences/sec
BERTScore Precision: 1.0000
BERTScore Recall:    1.0000
BERTScore F1:        1.0000


## 10. Compute MoverScore

MoverScore compares contextual word-embedding distributions using an Earth Mover's Distance-style alignment. It can be slow on the first run because pretrained model weights may be downloaded. The package is old, so dependency/API compatibility can vary across Colab runtimes.


In [10]:
# If import fails, keep the BLEU, ROUGE and BERTScore results and report
# that MoverScore could not run because of the package/runtime dependency issue.
try:
    from moverscore_v2 import get_idf_dict, word_mover_score

    idf_ref = get_idf_dict(references)
    idf_hyp = get_idf_dict(predictions)
    mover_scores = word_mover_score(
        references,
        predictions,
        idf_ref,
        idf_hyp,
        stop_words=[],
        n_gram=1,
        remove_subwords=True
    )
    print(f"MoverScore average: {float(np.mean(mover_scores)):.4f}")
except Exception as e:
    print("MoverScore could not run in this runtime.")
    print("Error:", repr(e))
    print("Try restarting the runtime and installing a compatible MoverScore/NumPy/PyEMD combination.")


MoverScore could not run in this runtime.
Error: ModuleNotFoundError("No module named 'pyemd'")
Try restarting the runtime and installing a compatible MoverScore/NumPy/PyEMD combination.


## 11. Inspect individual results

Review examples rather than relying only on one aggregate score.


In [11]:
import pandas as pd

results_df = pd.DataFrame({
    "Source (Hindi)": source_texts,
    "Reference (English)": references,
    "Model prediction": predictions
})
display(results_df.head(15))


,Source (Hindi),Reference (English),Model prediction
0,नमस्ते,hello,hello
1,आप कैसे हैं,how are you,how are you
2,मैं ठीक हूँ,i am fine,i am fine
3,मेरा नाम राहुल है,my name is rahul,my name is rahul
4,मैं एक छात्र हूँ,i am a student,i am a student
5,मुझे पानी चाहिए,i need water,i need water
6,धन्यवाद,thank you,thank you
7,शुभ प्रभात,good morning,good morning
8,शुभ रात्रि,good night,good night
9,मुझे खाना पसंद है,i like food,i like food


## 12. Conclusion / limitations

- The notebook implements a sequence-to-sequence LSTM with attention for Hindi → English translation.
- It computes corpus BLEU, average ROUGE-1/2/L F1, BERTScore Precision/Recall/F1, and attempts MoverScore.
- The parallel corpus is tiny and hand-written. Because the model is evaluated on training examples, the metrics are **demonstration values only** and must not be presented as a fair test-set evaluation.
- For a stronger experiment, use a real parallel corpus, split it into train/validation/test sets before training, and calculate all metrics only on the held-out test set.
